In [7]:
import argparse
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import matplotlib.pyplot as plt
import os
import random
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

#################################
# Parse Arguments
#################################
def parse_args():
    parser = argparse.ArgumentParser(
        description="Run CLS token drift detection experiments with a true domain shift."
    )
    parser.add_argument("--model_name", type=str,
                        default="nlpaueb/sec-bert-base",
                        help="HuggingFace model name (SEC-BERT variant).")
    parser.add_argument("--wiki_dataset_name", type=str, default="wikitext",
                        help="HuggingFace dataset for original text.")
    parser.add_argument("--wiki_dataset_config", type=str,
                        default="wikitext-2-raw-v1", help="Dataset config.")
    parser.add_argument("--wiki_split", type=str, default="train",
                        help="WikiText dataset split.")
    parser.add_argument("--financial_dataset_name", type=str, default="financial_phrasebank",
                        help="HuggingFace dataset for financial domain.")
    parser.add_argument("--financial_dataset_config", type=str,
                        default="sentences_50agree",
                        help="Configuration of financial_phrasebank.")
    parser.add_argument("--financial_split", type=str, default="train",
                        help="Financial dataset split.")
    parser.add_argument("--max_texts", type=int, default=30000,
                        help="Max number of texts to use from dataset.")
    parser.add_argument("--batch_size", type=int, default=64,
                        help="Batch size.")
    parser.add_argument("--drift_threshold_std", type=float, default=3.0,
                        help="Number of std devs for threshold.")
    parser.add_argument("--window_size", type=int, default=50,
                        help="Rolling window size for threshold computation.")
    parser.add_argument("--variance_threshold", type=float, default=0.0001,
                        help="Variance threshold for high cosine similarity detection.")
    parser.add_argument("--output_dir", type=str, default="results",
                        help="Directory to save results and plots.")
    args, unknown = parser.parse_known_args()
    return args

args = parse_args()

#################################
# Setup
#################################
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Using device:", device)

# Ensure output directory exists
os.makedirs(args.output_dir, exist_ok=True)

#################################
# Load Model and Tokenizer
#################################
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(args.model_name)
model = AutoModelForSequenceClassification.from_pretrained(args.model_name)
model.to(device)
model.eval()
print("Model ready on device:", device)

#################################
# Load Original (WikiText) Dataset
#################################
wiki_dataset = load_dataset(args.wiki_dataset_name, args.wiki_dataset_config,
                            split=args.wiki_split)
texts = wiki_dataset["text"]

if args.max_texts > 0 and args.max_texts < len(texts):
    texts = texts[:args.max_texts]
print(f"Original dataset loaded: {len(texts)} WikiText samples")

#################################
# Load Financial Dataset for Drift
#################################
fin_dataset = load_dataset(args.financial_dataset_name,
                           args.financial_dataset_config,
                           split=args.financial_split)
financial_texts = fin_dataset["sentence"]
random.shuffle(financial_texts)

# Simulate a second domain by using a slice of the financial texts as "Kaggle-like" data:
kaggle_texts = financial_texts[:1000]

print(f"Financial dataset loaded: {len(financial_texts)} samples")
print(f"Kaggle-like dataset loaded: {len(kaggle_texts)} samples")

#################################
# Simulate Multiple True Domain Drifts
#################################
n = len(texts)
drift_start_1, drift_end_1 = n // 3, n // 2
drift_start_2, drift_end_2 = 2 * n // 3, 5 * n // 6

fin_idx = 0
for i in range(drift_start_1, drift_end_1):
    texts[i] = financial_texts[fin_idx % len(financial_texts)]
    fin_idx += 1

kaggle_idx = 0
for i in range(drift_start_2, drift_end_2):
    texts[i] = kaggle_texts[kaggle_idx % len(kaggle_texts)]
    kaggle_idx += 1

print("Simulated multiple domain drifts:")
print(f"  Drift region 1: indices {drift_start_1} to {drift_end_1} (Finance).")
print(f"  Drift region 2: indices {drift_start_2} to {drift_end_2} (Kaggle).")

#################################
# Utility Functions
#################################
def batch_generator(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

def extract_cls_embeddings(model, tokenizer, texts, device):
    encodings = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)
    with torch.no_grad():
        outputs = model.bert(input_ids, attention_mask=attention_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
    return cls_embeddings.cpu().numpy()

#################################
# DriftDetector Class
#################################
class DriftDetector:
    def __init__(self, model, tokenizer, device, batch_generator, args):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.batch_generator = batch_generator
        self.args = args
        self.prototype = None
        self.prototypes = []
        self.drifts = []
        self.cosine_scores = []

    def _extract_embeddings(self, texts):
        """Extract embeddings for the provided texts."""
        embeddings = []
        for batch in self.batch_generator(texts, self.args.batch_size):
            embeddings.append(self._extract_cls_embeddings(batch))
        return np.concatenate(embeddings, axis=0)

    def _extract_cls_embeddings(self, batch):
        """Extract [CLS] embeddings using the model."""
        return extract_cls_embeddings(self.model, self.tokenizer, batch, self.device)

    def _calculate_threshold(self):
        """Calculate the drift detection threshold."""
        recent_scores = self.cosine_scores[-self.args.window_size:]
        return (np.mean(recent_scores) -
                self.args.drift_threshold_std * np.std(recent_scores))

    def _calculate_variance(self):
        """Calculate the variance of recent cosine similarity scores."""
        if len(self.cosine_scores) < self.args.window_size:
            return None
        recent_scores = self.cosine_scores[-self.args.window_size:]
        return np.var(recent_scores)

    def initialize_baseline(self, texts):
        """Initialize the baseline prototype from the given texts."""
        print("Initializing baseline prototype...")
        baseline_embeddings = self._extract_embeddings(texts[:self.args.max_texts])
        self.prototype = np.mean(baseline_embeddings, axis=0)
        self.prototypes.append(self.prototype)

    def detect_drifts(self, texts):
        """Process the dataset to detect drifts."""
        print("Processing dataset to detect drifts...")
        for i, batch in enumerate(tqdm(self.batch_generator(texts, self.args.batch_size))):
            batch_embeddings = self._extract_cls_embeddings(batch)
            if len(batch_embeddings.shape) == 1:
                batch_embeddings = batch_embeddings[np.newaxis, :]

            similarity = cosine_similarity([batch_embeddings.mean(axis=0)], [self.prototype])[0][0]
            self.cosine_scores.append(similarity)

            if len(self.cosine_scores) >= self.args.window_size:
                threshold = self._calculate_threshold()
                variance = self._calculate_variance()
                if similarity < threshold or (variance is not None and variance < self.args.variance_threshold):
                    drift_index = i * self.args.batch_size
                    self.drifts.append(drift_index)
                    print(f"Drift detected at batch {i}, index {drift_index}")

            self._update_prototype(batch_embeddings)

    def _update_prototype(self, batch_embeddings):
        """Update the prototype dynamically using batch embeddings."""
        delta = batch_embeddings - self.prototype
        weights = np.exp(-np.linalg.norm(delta, axis=1) / 2.0)
        self.prototype += np.sum(weights[:, None] * delta, axis=0) / np.sum(weights)
        self.prototypes.append(self.prototype)


#################################
# Detect Drifts and Plot Results
#################################
detector = DriftDetector(
    model=model,
    tokenizer=tokenizer,
    device=device,
    batch_generator=batch_generator,
    args=args
)

# Initialize baseline prototype
detector.initialize_baseline(texts)

# Detect drifts
detector.detect_drifts(texts)

Using device: mps
Loading model and tokenizer...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/sec-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model ready on device: mps
Original dataset loaded: 30000 WikiText samples
Financial dataset loaded: 4846 samples
Kaggle-like dataset loaded: 1000 samples
Simulated multiple domain drifts:
  Drift region 1: indices 10000 to 15000 (Finance).
  Drift region 2: indices 20000 to 25000 (Kaggle).
Initializing baseline prototype...
Processing dataset to detect drifts...


91it [02:26,  1.43s/it]

Drift detected at batch 90, index 5760


...

397it [07:04,  1.59s/it]

Drift detected at batch 396, index 25344


400it [07:11,  1.89s/it]

Drift detected at batch 399, index 25536


469it [09:05,  1.16s/it]


In [1]:
#################################
# Post-Processing:
# Cluster the drift indices
#################################
def cluster_drift_indices(
    drift_indices,
    distance_threshold=128,
    ignore_solo_outliers=False,
    min_cluster_size=2
):
    """
    Group consecutive drift indices that lie within `distance_threshold` of
    one another. Optionally ignore "solo" outliers if `ignore_solo_outliers=True`
    and cluster size is < min_cluster_size.
    """
    if not drift_indices:
        return []

    drift_indices = sorted(drift_indices)
    clusters = []
    current_cluster = [drift_indices[0]]

    for i in range(1, len(drift_indices)):
        if drift_indices[i] - drift_indices[i - 1] <= distance_threshold:
            current_cluster.append(drift_indices[i])
        else:
            clusters.append(current_cluster)
            current_cluster = [drift_indices[i]]
    clusters.append(current_cluster)

    # If ignoring outliers, remove clusters that don't meet min size
    if ignore_solo_outliers:
        clusters = [c for c in clusters if len(c) >= min_cluster_size]

    return clusters

def choose_cluster_representatives(clusters, mode="first"):
    """
    Given a list of drift index clusters, choose a single representative index
    per cluster. You can choose:
      - "first": use the first drift index in each cluster
      - "last": use the last drift index
      - "mean": use the average of the indices
      - "median": use the median of the indices
    """
    representatives = []
    for cluster in clusters:
        if mode == "first":
            representatives.append(cluster[0])
        elif mode == "last":
            representatives.append(cluster[-1])
        elif mode == "mean":
            representatives.append(int(sum(cluster) / len(cluster)))
        elif mode == "median":
            mid = len(cluster) // 2
            sorted_cluster = sorted(cluster)
            if len(cluster) % 2 == 1:
                representatives.append(sorted_cluster[mid])
            else:
                lower = sorted_cluster[mid - 1]
                upper = sorted_cluster[mid]
                representatives.append(int((lower + upper) / 2))
        else:
            raise ValueError(f"Unknown mode: {mode}")
    return representatives

# Now cluster the drift indices from the detector
drift_clusters = cluster_drift_indices(
    detector.drifts,
    distance_threshold=128,       # could be ~2 * batch_size if drifts are ~batch_size apart
    ignore_solo_outliers=False,   # set True to ignore small clusters
    min_cluster_size=2
)
drift_reps = choose_cluster_representatives(drift_clusters, mode="first")

print("All raw drift indices:", detector.drifts)
print("Clustered drifts:", drift_clusters)
print("Cluster representatives:", drift_reps)

#################################
# Plot Results
#################################
plt.figure(figsize=(12, 8))

# Plot the cosine similarity scores over time
plt.plot(detector.cosine_scores, label="Cosine Similarity", linewidth=2)

# Highlight known drift regions with shaded areas
plt.axvspan(drift_start_1 / args.batch_size, drift_end_1 / args.batch_size,
            color="orange", alpha=0.2, label="Drift Region 1 (Finance)")
plt.axvspan(drift_start_2 / args.batch_size, drift_end_2 / args.batch_size,
            color="blue", alpha=0.2, label="Drift Region 2 (Kaggle)")

# Plot cluster representatives as vertical lines (one line per cluster center)
for rep in drift_reps:
    # Convert from raw index to batch index
    batch_position = rep / args.batch_size
    if 0 <= batch_position < len(detector.cosine_scores):
        plt.axvline(batch_position, color="red", linestyle="--", alpha=0.8)
        plt.scatter(batch_position, detector.cosine_scores[int(batch_position)],
                    color="red", zorder=5)

plt.title("Cosine Similarity with Detected and Known Drifts", fontsize=16)
plt.xlabel("Batch Index", fontsize=14)
plt.ylabel("Cosine Similarity", fontsize=14)
plt.legend(loc="best", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

# Save and display the plot
plt.savefig(os.path.join(args.output_dir, "drift_detection_plot_with_highlights.png"))
plt.show()

print(f"\nFinal drift representatives (clustered): {drift_reps}")


NameError: name 'detector' is not defined

Detected drifts at indices: [5760, 7936, 13248, 13312, 13376, 13440, 13504, 13568, 13632, 13696, 13760, 13824, 13888, 13952, 14016, 14080, 14144, 14208, 14272, 14336, 14400, 14464, 14528, 14592, 14656, 14720, 14784, 14848, 14912, 14976, 15040, 15104, 15168, 15232, 15296, 15360, 15680, 15872, 15936, 20032, 23232, 23296, 23360, 23424, 23488, 23552, 23616, 23680, 23744, 23808, 23872, 23936, 24000, 24064, 24128, 24192, 24256, 24320, 24384, 24448, 24512, 24576, 24640, 24704, 24768, 24832, 24896, 24960, 25024, 25088, 25152, 25216, 25280, 25344, 25536]